In [2]:
import os
from google.colab import userdata

token = userdata.get("KAGGLE_API_TOKEN")
!mkdir -p ~/.kaggle
!echo "{token}" > ~/.kaggle/access_token
!chmod 600 ~/.kaggle/access_token

GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
!git clone https://{GITHUB_TOKEN}@github.com/amama22/walmart-forecasting
%cd walmart-forecasting

!kaggle competitions download \
  -c walmart-recruiting-store-sales-forecasting \
  -p data

import zipfile
data_dir = "data"
while any(f.endswith(".zip") for f in os.listdir(data_dir)):
    for fname in list(os.listdir(data_dir)):
        if fname.endswith(".zip"):
            fpath = os.path.join(data_dir, fname)
            with zipfile.ZipFile(fpath, "r") as z:
                z.extractall(data_dir)
            os.remove(fpath)

%pip install -q dagshub
!pip install mlflow --quiet

import dagshub
import mlflow

dagshub.init(repo_owner="amama22", repo_name="walmart-forecasting", mlflow=True)
mlflow.set_experiment("PatchTST_Training")

Cloning into 'walmart-forecasting'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (36/36), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 36 (delta 13), reused 19 (delta 5), pack-reused 0 (from 0)
Receiving objects: 100% (36/36), 215.58 KiB | 1.20 MiB/s, done.
Resolving deltas: 100% (13/13), done.
/content/walmart-forecasting/walmart-forecasting
100% 2.70M/2.70M [00:00<00:00, 248MB/s]



❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=0b977597-ccbd-4130-ad86-7e15f96ead6b&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=b13e776ff553cb9cc4a414c08da4a8c5e37892d762bb6f5b42bbc7fbb3296a92




Accessing as amama22

Initialized MLflow to track repo "amama22/walmart-forecasting"

Repository amama22/walmart-forecasting initialized!

<Experiment: artifact_location='mlflow-artifacts:/334563610a1943a09f91c8ec71ffcd36', creation_time=1783853197542, effective_trace_archival_retention=None, experiment_id='3', last_update_time=1783853197542, lifecycle_stage='active', name='PatchTST_Training', tags={}, trace_location=None, workspace='default'>

In [3]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

CUDA available: True
Device name: Tesla T4


In [4]:
%pip install -q neuralforecast

import numpy, torch, pytorch_lightning, neuralforecast
print("numpy:", numpy.__version__)
print("torch:", torch.__version__)
print("pytorch_lightning:", pytorch_lightning.__version__)
print("neuralforecast:", neuralforecast.__version__)
print("CUDA available:", torch.cuda.is_available())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.0/302.0 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 348.6/348.6 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 831.6/831.6 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 MB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 983.4/983.4 kB 72.0 MB/s eta 0:00:00
numpy: 2.0.2
torch: 2.11.0+cu128
pytorch_lightning: 2.5.6
neuralforecast: 3.2.0
CUDA available: True


In [5]:
import sys
sys.path.append(".")

import numpy as np
import pandas as pd
import mlflow

from src.data_prep import load_raw_data, merge_all
from src.evaluation import wmae

train, test, features, stores = load_raw_data(data_dir="data")
train_merged = merge_all(train, features, stores)

df = train_merged[["Store", "Dept", "Date", "Weekly_Sales", "IsHoliday"]].copy()
df["unique_id"] = df["Store"].astype(str) + "_" + df["Dept"].astype(str)
df = df.sort_values(["unique_id", "Date"])

HORIZON = 38

full_date_range = pd.date_range(df["Date"].min(), df["Date"].max(), freq="W-FRI")

def reindex_global(group):
    group = group.set_index("Date").reindex(full_date_range)
    group["Weekly_Sales"] = group["Weekly_Sales"].fillna(0)
    group["IsHoliday"] = group["IsHoliday"].astype("boolean").fillna(False).astype(bool)
    group.index.name = "Date"
    return group

df_global = (
    df.groupby("unique_id", group_keys=True)
    .apply(reindex_global, include_groups=False)
    .reset_index()
)

print("Shape after global reindex:", df_global.shape)
print("Unique series:", df_global["unique_id"].nunique())
print("All series same length:", df_global.groupby("unique_id").size().nunique() == 1)

with mlflow.start_run(run_name="PatchTST_Data_Validation"):
    mlflow.log_param("n_series", df_global["unique_id"].nunique())
    mlflow.log_param("series_length_weeks", df_global.groupby("unique_id").size().iloc[0])
    mlflow.log_param("reindex_fill_strategy", "zero_fill_gaps_and_pre_appearance")
    mlflow.log_metric("rows_after_reindex", df_global.shape[0])

Shape after global reindex: (476333, 6)
Unique series: 3331
All series same length: True
🏃 View run PatchTST_Data_Validation at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/d6b7622b3d1e4c09822242556f9932fd
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


In [6]:
from neuralforecast import NeuralForecast
from neuralforecast.models import PatchTST

nf_df = df_global.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["unique_id", "ds", "y"]]

subset_ids = nf_df["unique_id"].unique()[:20]
nf_df_subset = nf_df[nf_df["unique_id"].isin(subset_ids)]
print("Smoke subset shape:", nf_df_subset.shape, "series:", nf_df_subset["unique_id"].nunique())

smoke_model = PatchTST(
    h=HORIZON,
    input_size=104,
    max_steps=50,
    patch_len=16,
    stride=8,
    accelerator="gpu",
    devices=1,
    random_seed=42,
    enable_progress_bar=False,
    start_padding_enabled=True,
    dataloader_kwargs={"num_workers": 0},
    logger=False,
)

nf = NeuralForecast(models=[smoke_model], freq="W-FRI")
nf.fit(df=nf_df_subset)
preds = nf.predict()

print("Smoke test completed. Predictions shape:", preds.shape)
preds.head()

INFO:lightning_fabric.utilities.seed:Seed set to 42
/usr/local/lib/python3.12/dist-packages/neuralforecast/common/_base_model.py:749: UserWarning: val_check_steps is greater than max_steps, setting val_check_steps to max_steps.
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores


Smoke subset shape: (2860, 3) series: 20


INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 464 K  | train
------------------------------------------------------------------
464 K     Trainable params
3         Non-trainable params
464 K     Total params
1.858     Total estimated model params size (MB)
93        Modules in train mode
0         Modules in eval mode
/usr/local/lib/python3.12/dist-packages/pytorch_lightn

Smoke test completed. Predictions shape: (760, 3)


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


,unique_id,ds,PatchTST
0,10_1,2012-11-02,61620.367188
1,10_1,2012-11-09,58110.433594
2,10_1,2012-11-16,57137.875000
3,10_1,2012-11-23,65325.792969
4,10_1,2012-11-30,59892.628906


In [7]:
subset_ids_500 = nf_df["unique_id"].unique()[:500]
nf_df_500 = nf_df[nf_df["unique_id"].isin(subset_ids_500)]
print("500-series subset shape:", nf_df_500.shape, "series:", nf_df_500["unique_id"].nunique())

smoke_model_500 = PatchTST(
    h=HORIZON,
    input_size=104,
    max_steps=50,
    patch_len=16,
    stride=8,
    accelerator="gpu",
    devices=1,
    random_seed=42,
    enable_progress_bar=False,
    start_padding_enabled=True,
    dataloader_kwargs={"num_workers": 0},
    logger=False,
)

nf_500 = NeuralForecast(models=[smoke_model_500], freq="W-FRI")
nf_500.fit(df=nf_df_500)
preds_500 = nf_500.predict()

print("500-series smoke test completed. Predictions shape:", preds_500.shape)

INFO:lightning_fabric.utilities.seed:Seed set to 42
/usr/local/lib/python3.12/dist-packages/neuralforecast/common/_base_model.py:749: UserWarning: val_check_steps is greater than max_steps, setting val_check_steps to max_steps.
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              |

500-series subset shape: (71500, 3) series: 500


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=50` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


500-series smoke test completed. Predictions shape: (19000, 3)


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


In [8]:
smoke_model_full = PatchTST(
    h=HORIZON,
    input_size=104,
    max_steps=50,
    patch_len=16,
    stride=8,
    accelerator="gpu",
    devices=1,
    random_seed=42,
    enable_progress_bar=False,
    start_padding_enabled=True,
    dataloader_kwargs={"num_workers": 0},
    logger=False,
)

nf_full = NeuralForecast(models=[smoke_model_full], freq="W-FRI")
nf_full.fit(df=nf_df)
preds_full = nf_full.predict()

print("Full-scale smoke test completed. Predictions shape:", preds_full.shape)
print("Expected:", nf_df['unique_id'].nunique() * HORIZON)

INFO:lightning_fabric.utilities.seed:Seed set to 42
/usr/local/lib/python3.12/dist-packages/neuralforecast/common/_base_model.py:749: UserWarning: val_check_steps is greater than max_steps, setting val_check_steps to max_steps.
  warnings.warn(
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              |

Full-scale smoke test completed. Predictions shape: (126578, 3)
Expected: 126578


In [9]:
nf_df_holiday = df_global.rename(columns={"Date": "ds", "Weekly_Sales": "y"})[["unique_id", "ds", "y", "IsHoliday"]]

with mlflow.start_run(run_name="PatchTST_Baseline"):
    mlflow.log_param("input_size", 104)
    mlflow.log_param("horizon", HORIZON)
    mlflow.log_param("max_steps", 500)
    mlflow.log_param("n_windows", 1)
    mlflow.log_param("patch_len", 16)
    mlflow.log_param("stride", 8)

    model = PatchTST(
        h=HORIZON,
        input_size=104,
        max_steps=500,
        patch_len=16,
        stride=8,
        accelerator="gpu",
        devices=1,
        random_seed=42,
        enable_progress_bar=False,
        start_padding_enabled=True,
        dataloader_kwargs={"num_workers": 0},
        logger=False,
    )

    nf = NeuralForecast(models=[model], freq="W-FRI")
    cv_df = nf.cross_validation(df=nf_df_holiday[["unique_id", "ds", "y"]], n_windows=1, step_size=HORIZON)
    cv_df = cv_df.merge(nf_df_holiday[["unique_id", "ds", "IsHoliday"]], on=["unique_id", "ds"], how="left")

    baseline_wmae = wmae(cv_df["y"], cv_df["PatchTST"], cv_df["IsHoliday"])
    mlflow.log_metric("mean_wmae", baseline_wmae)
    print(f"PatchTST baseline WMAE (1 fold, 500 steps): {baseline_wmae:.2f}")

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 464 K  | train
------------------------------------------------------------------
464 K     Trainable params
3 

PatchTST baseline WMAE (1 fold, 500 steps): 1802.91
🏃 View run PatchTST_Baseline at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/f03914ab3e364594b92dd13c492bd902
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


In [10]:
%pip install -q optuna
import optuna

def objective(trial):
    input_size = trial.suggest_int("input_size", 26, 66)
    patch_len = trial.suggest_categorical("patch_len", [8, 16, 24, 32])
    stride = trial.suggest_categorical("stride", [4, 8, 16])
    hidden_size = trial.suggest_categorical("hidden_size", [64, 128, 256])
    n_heads = trial.suggest_categorical("n_heads", [2, 4, 8])
    lr = trial.suggest_float("lr", 1e-4, 1e-2, log=True)

    model = PatchTST(
        h=HORIZON,
        input_size=input_size,
        max_steps=300,
        patch_len=patch_len,
        stride=stride,
        hidden_size=hidden_size,
        n_heads=n_heads,
        learning_rate=lr,
        accelerator="gpu",
        devices=1,
        random_seed=42,
        enable_progress_bar=False,
        start_padding_enabled=True,
        dataloader_kwargs={"num_workers": 0},
        logger=False,
    )

    nf_trial = NeuralForecast(models=[model], freq="W-FRI")
    cv_df = nf_trial.cross_validation(df=nf_df_holiday[["unique_id", "ds", "y"]], n_windows=1, step_size=HORIZON)
    cv_df = cv_df.merge(nf_df_holiday[["unique_id", "ds", "IsHoliday"]], on=["unique_id", "ds"], how="left")

    trial_wmae = wmae(cv_df["y"], cv_df["PatchTST"], cv_df["IsHoliday"])

    with mlflow.start_run(run_name=f"trial_{trial.number}", nested=True):
        mlflow.log_params({"input_size": input_size, "patch_len": patch_len, "stride": stride, "hidden_size": hidden_size, "n_heads": n_heads, "lr": lr, "max_steps": 300})
        mlflow.log_metric("wmae", trial_wmae)

    return trial_wmae

with mlflow.start_run(run_name="PatchTST_HPO"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=20)

    mlflow.log_params(study.best_params)
    mlflow.log_metric("best_wmae", study.best_value)
    print("Best WMAE:", study.best_value)
    print("Best params:", study.best_params)

[I 2026-07-12 10:58:42,767] A new study created in memory with name: no-name-56eee80a-c818-40a9-912a-cbdd79978eb2
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone

🏃 View run trial_0 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/8a1fefc95a2c4d72b03b9ae51d577103
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 1.3 M  | train
------------------------------------------------------------------
1.3 M     Trainable params
3         Non-trainable params
1.3 M     Total params


🏃 View run trial_1 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/d565df70ed7f4d3aaa7a4fd1920b01ad
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 1.2 M  | train
------------------------------------------------------------------
1.2 M     Trainable params
3         Non-trainable params
1.2 M     Total params


...

🏃 View run trial_18 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/f6fc96e65e7a4590886801a63a7ba35f
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


[I 2026-07-12 11:02:17,367] Trial 18 finished with value: 1621.7653212278885 and parameters: {'input_size': 55, 'patch_len': 16, 'stride': 16, 'hidden_size': 256, 'n_heads': 2, 'lr': 0.0002864779912485647}. Best is trial 11 with value: 1555.8936745695892.
INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train       

🏃 View run trial_19 at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/dc189371f3b6437db82045940cfcbdde
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


[I 2026-07-12 11:02:30,188] Trial 19 finished with value: 1650.1767998563244 and parameters: {'input_size': 62, 'patch_len': 16, 'stride': 8, 'hidden_size': 256, 'n_heads': 4, 'lr': 0.0011522569180238591}. Best is trial 11 with value: 1555.8936745695892.


Best WMAE: 1555.8936745695892
Best params: {'input_size': 54, 'patch_len': 16, 'stride': 4, 'hidden_size': 256, 'n_heads': 2, 'lr': 0.0006580239028512152}
🏃 View run PatchTST_HPO at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/3d611b17a29b4658a0bc36a7e2046f80
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


In [11]:
best_params = {
    "input_size": 54,
    "patch_len": 16,
    "stride": 4,
    "hidden_size": 256,
    "n_heads": 2,
    "lr": 0.0006580239028512152,
}

for steps in [300, 800]:
    model = PatchTST(
        h=HORIZON,
        input_size=best_params["input_size"],
        max_steps=steps,
        patch_len=best_params["patch_len"],
        stride=best_params["stride"],
        hidden_size=best_params["hidden_size"],
        n_heads=best_params["n_heads"],
        learning_rate=best_params["lr"],
        accelerator="gpu",
        devices=1,
        random_seed=42,
        enable_progress_bar=False,
        start_padding_enabled=True,
        dataloader_kwargs={"num_workers": 0},
        logger=False,
    )
    nf_check = NeuralForecast(models=[model], freq="W-FRI")
    cv_df = nf_check.cross_validation(df=nf_df_holiday[["unique_id", "ds", "y"]], n_windows=1, step_size=HORIZON)
    cv_df = cv_df.merge(nf_df_holiday[["unique_id", "ds", "IsHoliday"]], on=["unique_id", "ds"], how="left")
    step_wmae = wmae(cv_df["y"], cv_df["PatchTST"], cv_df["IsHoliday"])
    print(f"max_steps={steps}: WMAE = {step_wmae:.2f}")

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 1.3 M  | train
------------------------------------------------------------------
1.3 M     Trainable params
3 

max_steps=300: WMAE = 1555.89


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_steps=800` reached.
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


max_steps=800: WMAE = 1609.30


In [12]:
with mlflow.start_run(run_name="PatchTST_Final_Model"):
    mlflow.log_params(best_params)
    mlflow.log_param("max_steps", 300)
    mlflow.log_param("note", "800-step refit scored worse (1609.30) than 300-step HPO config (1555.89) -- kept 300")

    final_model = PatchTST(
        h=HORIZON,
        input_size=best_params["input_size"],
        max_steps=300,
        patch_len=best_params["patch_len"],
        stride=best_params["stride"],
        hidden_size=best_params["hidden_size"],
        n_heads=best_params["n_heads"],
        learning_rate=best_params["lr"],
        accelerator="gpu",
        devices=1,
        random_seed=42,
        enable_progress_bar=False,
        start_padding_enabled=True,
        dataloader_kwargs={"num_workers": 0},
        logger=False,
    )

    nf_final = NeuralForecast(models=[final_model], freq="W-FRI")
    cv_df = nf_final.cross_validation(df=nf_df_holiday[["unique_id", "ds", "y"]], n_windows=1, step_size=HORIZON)
    cv_df = cv_df.merge(nf_df_holiday[["unique_id", "ds", "IsHoliday"]], on=["unique_id", "ds"], how="left")
    final_wmae = wmae(cv_df["y"], cv_df["PatchTST"], cv_df["IsHoliday"])
    mlflow.log_metric("final_wmae", final_wmae)
    print(f"Final PatchTST WMAE (300 steps, held-out fold): {final_wmae:.2f}")

    nf_final.save(path="./nbeats_patchtst_model", overwrite=True)

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 1.3 M  | train
------------------------------------------------------------------
1.3 M     Trainable params
3 

Final PatchTST WMAE (300 steps, held-out fold): 1555.89
🏃 View run PatchTST_Final_Model at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/ab59f74154ff4946ba908c89fabfffc1
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


In [13]:
import mlflow.pyfunc

class PatchTSTWrapper(mlflow.pyfunc.PythonModel):
    def load_context(self, context):
        from neuralforecast import NeuralForecast
        self.nf = NeuralForecast.load(path=context.artifacts["model_dir"])

    def predict(self, context, model_input):
        preds = self.nf.predict(df=model_input)
        return preds


with mlflow.start_run(run_name="PatchTST_Registration"):
    mlflow.log_params(best_params)
    mlflow.log_param("max_steps", 300)
    mlflow.log_metric("held_out_wmae", final_wmae)

    mlflow.pyfunc.log_model(
        artifact_path="patchtst_pipeline",
        python_model=PatchTSTWrapper(),
        artifacts={"model_dir": "./nbeats_patchtst_model"},
        registered_model_name="walmart-patchtst-store-sales",
    )

    print("Logged and registered as 'walmart-patchtst-store-sales'")
    print(f"Held-out fold WMAE: {final_wmae:.2f}")

/usr/local/lib/python3.12/dist-packages/mlflow/pyfunc/utils/data_validation.py:187: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(
2026/07/12 11:10:01 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/12 11:10:01 WARNING mlflow.pyfunc: Passing a Python object as `python_model` causes it to be serialized using CloudPickle, it requires exercising caution as Python object serialization mechanisms may execute arbitrary code during deserialization.Consider using a file path (str or Path) instead. See https://mlflow.org/docs/latest/ml/model/models-from-code/ for details.


2026/07/12 11:10:21 WARNING mlflow.utils.requirements_utils: Found torchvision version (0.26.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torchvision==0.26.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.
Successfully registered model 'walmart-patchtst-store-sales'.
2026/07/12 11:10:26 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: walmart-patchtst-store-sales, version 1
Created version '1' of model 'walmart-patchtst-store-sales'.


Logged and registered as 'walmart-patchtst-store-sales'
Held-out fold WMAE: 1555.89
🏃 View run PatchTST_Registration at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/3ab76ebb0e2944f99130f43ab9c0a5fb
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


In [14]:
test["unique_id"] = test["Store"].astype(str) + "_" + test["Dept"].astype(str)

with mlflow.start_run(run_name="PatchTST_Submission_Refit"):
    mlflow.log_params(best_params)
    mlflow.log_param("max_steps", 300)
    mlflow.log_param("note", "refit on full train history for Kaggle submission")

    submission_model = PatchTST(
        h=39,
        input_size=best_params["input_size"],
        max_steps=300,
        patch_len=best_params["patch_len"],
        stride=best_params["stride"],
        hidden_size=best_params["hidden_size"],
        n_heads=best_params["n_heads"],
        learning_rate=best_params["lr"],
        accelerator="gpu",
        devices=1,
        random_seed=42,
        enable_progress_bar=False,
        start_padding_enabled=True,
        dataloader_kwargs={"num_workers": 0},
        logger=False,
    )

    nf_submit = NeuralForecast(models=[submission_model], freq="W-FRI")
    nf_submit.fit(df=nf_df)  # full history, no truncation

test_preds = nf_submit.predict()

pred_lookup = {}
for _, row in test_preds.iterrows():
    store, dept = row["unique_id"].split("_")
    pred_lookup[(store, dept, row["ds"].strftime("%Y-%m-%d"))] = max(row["PatchTST"], 0)

rows = []
missing_count = 0
for _, row in test.iterrows():
    key = (str(row["Store"]), str(row["Dept"]), row["Date"].strftime("%Y-%m-%d"))
    pred_value = pred_lookup.get(key)
    if pred_value is None:
        pred_value = 0.0
        missing_count += 1
    rows.append({"Id": f"{key[0]}_{key[1]}_{key[2]}", "Weekly_Sales": pred_value})

patchtst_submission = pd.DataFrame(rows)
print("Submission shape:", patchtst_submission.shape, "| expected:", test.shape[0])
print("Rows filled with fallback 0 (unseen series):", missing_count)
patchtst_submission.to_csv("submission_patchtst.csv", index=False)
patchtst_submission.head()

INFO:lightning_fabric.utilities.seed:Seed set to 42
INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name                | Type              | Params | Mode 
------------------------------------------------------------------
0 | loss                | MAE               | 0      | train
1 | hist_cat_embeddings | ModuleList        | 0      | train
2 | futr_cat_embeddings | ModuleList        | 0      | train
3 | stat_cat_embeddings | ModuleList        | 0      | train
4 | padder_train        | ConstantPad1d     | 0      | train
5 | scaler              | TemporalNorm      | 0      | train
6 | model               | PatchTST_backbone | 1.3 M  | train
------------------------------------------------------------------
1.3 M     Trainable params
3 

🏃 View run PatchTST_Submission_Refit at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3/runs/36bb9b1d060d47ca9dd130044eb07670
🧪 View experiment at: https://dagshub.com/amama22/walmart-forecasting.mlflow/#/experiments/3


INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Submission shape: (115064, 2) | expected: 115064
Rows filled with fallback 0 (unseen series): 36


,Id,Weekly_Sales
0,1_1_2012-11-02,26715.470703
1,1_1_2012-11-09,22991.947266
2,1_1_2012-11-16,20296.197266
3,1_1_2012-11-23,20399.957031
4,1_1_2012-11-30,23365.050781


In [15]:
!kaggle competitions submit -c walmart-recruiting-store-sales-forecasting -f submission_patchtst.csv -m "PatchTST, neuralforecast, tuned, WMAE~1556 single-fold"

100% 3.56M/3.56M [00:00<00:00, 5.17MB/s]
Successfully submitted to Walmart Recruiting - Store Sales Forecasting